# SHOT 3-Stage Heatmap Training v2
**946장 핸드폰 데이터 + 8,841장 방송 데이터**

- Stage 1: Broadcast pretrain
- Stage 2: Mixed 50:50 oversampling
- Stage 3: Phone fine-tune

⚠️ Runtime > Change runtime type > **T4 GPU** 선택

## 0. Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/SHOT', exist_ok=True)
os.chdir('/content/SHOT')
print('Working dir:', os.getcwd())

In [ ]:
# Install dependencies
!pip install -q albumentations timm
# Clone repo for model code
!git clone https://github.com/kokoro456/SHOT-AI.git repo 2>/dev/null || (cd repo && git pull)
import sys
sys.path.insert(0, '/content/SHOT/repo/ml/src')

## 1. Data Preparation

In [ ]:
# === Phone data (946 images) ===
# phone_256.zip Google Drive file ID
PHONE_ZIP_ID = '1aAR3W69SOTJVLrr2gLCqa6OqgxPhfCFs'

import shutil

if not os.path.exists('phone_256'):
    # Try Drive path first
    drive_path = '/content/drive/MyDrive/SHOT-AI/phone_256.zip'
    if os.path.exists(drive_path):
        print('Copying from Drive...')
        shutil.copy(drive_path, 'phone_256.zip')
    else:
        # Download by file ID
        print('Downloading from Drive by ID...')
        !gdown --id {PHONE_ZIP_ID} -O phone_256.zip 2>/dev/null || \
            !pip install -q gdown && gdown --id {PHONE_ZIP_ID} -O phone_256.zip
    
    !unzip -q phone_256.zip
    print(f'Phone images: {len(os.listdir("phone_256"))}')
else:
    print(f'Phone images already extracted: {len(os.listdir("phone_256"))}')

In [ ]:
# === Broadcast data (8,841 images) ===
# Download from Kaggle if not cached
if not os.path.exists('broadcast_256'):
    print('Preparing broadcast data...')
    
    # Check if broadcast data exists in Drive
    drive_broadcast = '/content/drive/MyDrive/SHOT-AI/broadcast_256.zip'
    if os.path.exists(drive_broadcast):
        print('Copying broadcast from Drive...')
        shutil.copy(drive_broadcast, 'broadcast_256.zip')
        !unzip -q broadcast_256.zip
    else:
        print('Need to prepare broadcast data from Kaggle.')
        print('Run: pip install kaggle && kaggle datasets download yastrebksv/tennis-court-detector')
        print('Or upload broadcast_256.zip to Drive/SHOT-AI/')
        # Alternative: download and preprocess
        !pip install -q kaggle
        # You may need to upload kaggle.json first
        !kaggle datasets download yastrebksv/tennis-court-detector -p /content/SHOT/kaggle_data --unzip 2>/dev/null
        
        if os.path.exists('kaggle_data'):
            # Convert and resize
            from repo.ml.src.convert_dataset import main as convert_main
            # Manual conversion
            print('Converting broadcast dataset...')
            !cd repo && python ml/src/convert_dataset.py \
                --input /content/SHOT/kaggle_data/tennis_court \
                --output /content/SHOT/broadcast_annotations.json \
                --copy-images --image-dir /content/SHOT/broadcast_raw
            
            # Resize to 256x256
            os.makedirs('broadcast_256', exist_ok=True)
            from PIL import Image
            from pathlib import Path
            for f in sorted(Path('broadcast_raw').glob('*.jpg')):
                img = Image.open(f).convert('RGB').resize((256,256), Image.LANCZOS)
                img.save(f'broadcast_256/{f.name}', 'JPEG', quality=90)
            print(f'Broadcast images: {len(os.listdir("broadcast_256"))}')
            
            # Save to Drive for next time
            !cd /content/SHOT && zip -qr broadcast_256.zip broadcast_256/ broadcast_annotations.json
            os.makedirs('/content/drive/MyDrive/SHOT-AI', exist_ok=True)
            shutil.copy('broadcast_256.zip', drive_broadcast)
            print('Saved broadcast_256.zip to Drive')
else:
    print(f'Broadcast images already extracted: {len(os.listdir("broadcast_256"))}')

In [ ]:
# Verify data
import json

with open('phone_combined_final.json') as f:
    phone_data = json.load(f)

# Check broadcast annotations
broadcast_ann = None
for p in ['broadcast_annotations.json', 'repo/ml/data/raw/annotations.json']:
    if os.path.exists(p):
        with open(p) as f:
            broadcast_ann = json.load(f)
        break

print(f'Phone data: {len(phone_data)} images')
if broadcast_ann:
    print(f'Broadcast data: {len(broadcast_ann)} images')
else:
    print('WARNING: Broadcast annotations not found!')

# Quick validation
from pathlib import Path
phone_found = sum(1 for x in phone_data if Path(f'phone_256/{x["image"]}').exists())
print(f'Phone images found: {phone_found}/{len(phone_data)}')

## 2. Model & Dataset Definition

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import numpy as np
from PIL import Image
from pathlib import Path
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')

In [ ]:
# Heatmap Model (same as model_heatmap.py)
class CourtKeypointHeatmapModel(nn.Module):
    def __init__(self, num_keypoints=8, heatmap_size=64):
        super().__init__()
        self.num_keypoints = num_keypoints
        self.heatmap_size = heatmap_size
        
        # MobileNetV3-Small backbone
        self.backbone = timm.create_model('mobilenetv3_small_100', pretrained=True, features_only=True)
        
        # Get channel info
        with torch.no_grad():
            dummy = torch.randn(1, 3, 256, 256)
            features = self.backbone(dummy)
            channels = [f.shape[1] for f in features]
        
        # Decoder with skip connections
        self.up4 = nn.Sequential(
            nn.ConvTranspose2d(channels[-1], 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True)
        )
        self.up3 = nn.Sequential(
            nn.ConvTranspose2d(128 + channels[-2], 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True)
        )
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(64 + channels[-3], 32, 4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True)
        )
        
        self.head = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, num_keypoints, 1)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        
        x = self.up4(features[-1])
        x = torch.cat([x, F.interpolate(features[-2], size=x.shape[2:], mode='bilinear', align_corners=False)], dim=1)
        x = self.up3(x)
        x = torch.cat([x, F.interpolate(features[-3], size=x.shape[2:], mode='bilinear', align_corners=False)], dim=1)
        x = self.up2(x)
        
        heatmaps = self.head(x)
        heatmaps = F.interpolate(heatmaps, size=(self.heatmap_size, self.heatmap_size), mode='bilinear', align_corners=False)
        return heatmaps

In [ ]:
# Dataset
KEYPOINT_IDS = [9, 10, 11, 12, 13, 14, 15, 16]

def generate_heatmap(cx, cy, size=64, sigma=2.0):
    x = torch.arange(size, dtype=torch.float32)
    y = torch.arange(size, dtype=torch.float32)
    yy, xx = torch.meshgrid(y, x, indexing='ij')
    heatmap = torch.exp(-((xx - cx)**2 + (yy - cy)**2) / (2 * sigma**2))
    return heatmap

class CourtDataset(Dataset):
    def __init__(self, annotations, image_dir, transform=None, heatmap_size=64, sigma=2.0):
        self.annotations = annotations
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.heatmap_size = heatmap_size
        self.sigma = sigma
    
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, idx):
        item = self.annotations[idx]
        img_path = self.image_dir / item['image']
        image = np.array(Image.open(img_path).convert('RGB'))
        
        # Apply augmentation (image only, no keypoints)
        if self.transform:
            transformed = self.transform(image=image)
            image = transformed['image']
        
        # Convert to tensor and normalize
        if isinstance(image, np.ndarray):
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        image = (image - mean) / std
        
        # Generate heatmaps from normalized coordinates
        heatmaps = torch.zeros(len(KEYPOINT_IDS), self.heatmap_size, self.heatmap_size)
        for i, kp_id in enumerate(KEYPOINT_IDS):
            kp = item['keypoints'].get(str(kp_id), {})
            if kp.get('visible', True) and kp.get('x', 0) > 0:
                cx = kp['x'] * self.heatmap_size
                cy = kp['y'] * self.heatmap_size
                heatmaps[i] = generate_heatmap(cx, cy, self.heatmap_size, self.sigma)
        
        return image, heatmaps

In [ ]:
# Augmentations
train_transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.0),  # Don't flip - keypoints are ordered L/R
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=0.5),
    A.GaussianBlur(blur_limit=(3, 7), p=0.2),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
    A.ImageCompression(quality_range=(50, 95), p=0.3),
])

val_transform = A.Compose([
    A.Resize(256, 256),
])

## 3. Training Functions

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

def decode_heatmap(heatmaps):
    """Decode heatmap to keypoint coordinates using argmax + sub-pixel refinement."""
    B, C, H, W = heatmaps.shape
    heatmaps_flat = heatmaps.view(B, C, -1)
    max_idx = heatmaps_flat.argmax(dim=2)
    
    cy = (max_idx // W).float()
    cx = (max_idx % W).float()
    
    return torch.stack([cx, cy], dim=2)  # [B, C, 2]

def compute_keypoint_error(pred_heatmaps, gt_heatmaps, heatmap_size=64):
    """Compute mean pixel error between predicted and GT keypoint positions."""
    pred_coords = decode_heatmap(pred_heatmaps)  # [B, 8, 2]
    gt_coords = decode_heatmap(gt_heatmaps)  # [B, 8, 2]
    
    # Only compute error for visible keypoints (GT heatmap max > 0.1)
    gt_max = gt_heatmaps.view(gt_heatmaps.shape[0], gt_heatmaps.shape[1], -1).max(dim=2)[0]
    visible_mask = gt_max > 0.1  # [B, 8]
    
    errors = torch.sqrt(((pred_coords - gt_coords) ** 2).sum(dim=2))  # [B, 8]
    
    if visible_mask.sum() > 0:
        mean_error = (errors * visible_mask.float()).sum() / visible_mask.float().sum()
    else:
        mean_error = errors.mean()
    
    return mean_error

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    total_error = 0
    n_batches = 0
    
    for images, heatmaps in loader:
        images = images.to(device)
        heatmaps = heatmaps.to(device)
        
        pred = model(images)
        loss = criterion(pred, heatmaps)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        with torch.no_grad():
            total_error += compute_keypoint_error(pred, heatmaps).item()
        n_batches += 1
    
    return total_loss / n_batches, total_error / n_batches

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    total_error = 0
    n_batches = 0
    
    for images, heatmaps in loader:
        images = images.to(device)
        heatmaps = heatmaps.to(device)
        
        pred = model(images)
        loss = criterion(pred, heatmaps)
        
        total_loss += loss.item()
        total_error += compute_keypoint_error(pred, heatmaps).item()
        n_batches += 1
    
    return total_loss / n_batches, total_error / n_batches

def train_stage(model, train_loader, val_loader, optimizer, scheduler, criterion,
                device, epochs, stage_name, save_dir, patience=10):
    """Train one stage with early stopping."""
    best_val_error = float('inf')
    best_epoch = 0
    no_improve = 0
    
    os.makedirs(save_dir, exist_ok=True)
    
    print(f'\n{"="*60}')
    print(f'  {stage_name}')
    print(f'  Train: {len(train_loader.dataset)}, Val: {len(val_loader.dataset)}')
    print(f'{"="*60}')
    
    for epoch in range(epochs):
        t0 = time.time()
        train_loss, train_error = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_error = evaluate(model, val_loader, criterion, device)
        
        if scheduler:
            scheduler.step()
        
        elapsed = time.time() - t0
        lr = optimizer.param_groups[0]['lr']
        
        improved = ''
        if val_error < best_val_error:
            best_val_error = val_error
            best_epoch = epoch + 1
            no_improve = 0
            torch.save(model.state_dict(), f'{save_dir}/best.pth')
            improved = ' ⭐ BEST'
        else:
            no_improve += 1
        
        print(f'  Epoch {epoch+1:3d}/{epochs} | '
              f'Train: {train_loss:.4f} ({train_error:.2f}px) | '
              f'Val: {val_loss:.4f} ({val_error:.2f}px) | '
              f'LR: {lr:.6f} | {elapsed:.0f}s{improved}')
        
        if no_improve >= patience:
            print(f'  Early stopping at epoch {epoch+1} (best: epoch {best_epoch}, {best_val_error:.2f}px)')
            break
    
    # Load best model
    model.load_state_dict(torch.load(f'{save_dir}/best.pth'))
    print(f'  ✅ {stage_name} complete. Best: epoch {best_epoch}, {best_val_error:.2f}px')
    
    # Save to Drive
    drive_dir = f'/content/drive/MyDrive/SHOT-AI/models/3stage_v2'
    os.makedirs(drive_dir, exist_ok=True)
    shutil.copy(f'{save_dir}/best.pth', f'{drive_dir}/{stage_name.lower().replace(" ", "_")}_best.pth')
    print(f'  Saved to Drive: {drive_dir}/')
    
    return best_val_error

## 4. Prepare Data Splits

In [ ]:
import random

# Phone data: 80/20 split
random.seed(42)
phone_shuffled = phone_data.copy()
random.shuffle(phone_shuffled)
split = int(len(phone_shuffled) * 0.8)
phone_train = phone_shuffled[:split]
phone_val = phone_shuffled[split:]

print(f'Phone train: {len(phone_train)}, Phone val: {len(phone_val)}')

# Broadcast data (if available)
if broadcast_ann:
    random.shuffle(broadcast_ann)
    bc_split = int(len(broadcast_ann) * 0.8)
    broadcast_train = broadcast_ann[:bc_split]
    broadcast_val = broadcast_ann[bc_split:]
    print(f'Broadcast train: {len(broadcast_train)}, Broadcast val: {len(broadcast_val)}')
else:
    broadcast_train = []
    broadcast_val = []
    print('No broadcast data - will skip Stage 1')

## 5. Stage 1: Broadcast Pretrain

In [ ]:
model = CourtKeypointHeatmapModel(num_keypoints=8, heatmap_size=64).to(device)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

criterion = nn.MSELoss()

if len(broadcast_train) > 0:
    bc_train_ds = CourtDataset(broadcast_train, 'broadcast_256', train_transform)
    bc_val_ds = CourtDataset(broadcast_val, 'broadcast_256', val_transform)
    
    bc_train_loader = DataLoader(bc_train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
    bc_val_loader = DataLoader(bc_val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
    
    optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=30)
    
    stage1_error = train_stage(
        model, bc_train_loader, bc_val_loader, optimizer, scheduler, criterion,
        device, epochs=30, stage_name='Stage1_Broadcast', save_dir='checkpoints/stage1', patience=10
    )
else:
    print('Skipping Stage 1 (no broadcast data)')
    stage1_error = None

## 6. Stage 2: Mixed Training (50:50 Oversampling)

In [ ]:
# Combined dataset with oversampling
if len(broadcast_train) > 0:
    # Oversample phone data to match broadcast
    oversample_ratio = len(broadcast_train) // len(phone_train) + 1
    mixed_train = broadcast_train + phone_train * oversample_ratio
    mixed_train = mixed_train[:len(broadcast_train) * 2]  # Cap at 2x broadcast
    
    # Create dataset with mixed image dirs
    # We need a custom dataset that handles both dirs
    class MixedDataset(Dataset):
        def __init__(self, annotations, transform=None, heatmap_size=64, sigma=2.0):
            self.annotations = annotations
            self.transform = transform
            self.heatmap_size = heatmap_size
            self.sigma = sigma
        
        def __len__(self):
            return len(self.annotations)
        
        def __getitem__(self, idx):
            item = self.annotations[idx]
            
            # Determine image directory based on source
            source = item.get('source', '')
            if source in ('youtube', 'sntc', 'sntc_selected'):
                img_dir = 'phone_256'
            else:
                img_dir = 'broadcast_256'
            
            img_path = Path(img_dir) / item['image']
            if not img_path.exists():
                # Fallback
                for d in ['phone_256', 'broadcast_256']:
                    p = Path(d) / item['image']
                    if p.exists():
                        img_path = p
                        break
            
            image = np.array(Image.open(img_path).convert('RGB'))
            
            if self.transform:
                transformed = self.transform(image=image)
                image = transformed['image']
            
            if isinstance(image, np.ndarray):
                image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
            image = (image - mean) / std
            
            heatmaps = torch.zeros(len(KEYPOINT_IDS), self.heatmap_size, self.heatmap_size)
            for i, kp_id in enumerate(KEYPOINT_IDS):
                kp = item['keypoints'].get(str(kp_id), {})
                if kp.get('visible', True) and kp.get('x', 0) > 0:
                    cx = kp['x'] * self.heatmap_size
                    cy = kp['y'] * self.heatmap_size
                    heatmaps[i] = generate_heatmap(cx, cy, self.heatmap_size, self.sigma)
            
            return image, heatmaps
    
    mixed_train_ds = MixedDataset(mixed_train, train_transform)
    # Val: phone only (target domain)
    phone_val_ds = CourtDataset(phone_val, 'phone_256', val_transform)
    
    mixed_train_loader = DataLoader(mixed_train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
    phone_val_loader = DataLoader(phone_val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
else:
    # No broadcast - just use phone data
    phone_train_ds = CourtDataset(phone_train, 'phone_256', train_transform)
    phone_val_ds = CourtDataset(phone_val, 'phone_256', val_transform)
    
    mixed_train_loader = DataLoader(phone_train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
    phone_val_loader = DataLoader(phone_val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

optimizer = AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=40)

stage2_error = train_stage(
    model, mixed_train_loader, phone_val_loader, optimizer, scheduler, criterion,
    device, epochs=40, stage_name='Stage2_Mixed', save_dir='checkpoints/stage2', patience=15
)

## 7. Stage 3: Phone Fine-tune

In [ ]:
phone_train_ds = CourtDataset(phone_train, 'phone_256', train_transform)
phone_val_ds = CourtDataset(phone_val, 'phone_256', val_transform)

phone_train_loader = DataLoader(phone_train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
phone_val_loader2 = DataLoader(phone_val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=30)

stage3_error = train_stage(
    model, phone_train_loader, phone_val_loader2, optimizer, scheduler, criterion,
    device, epochs=30, stage_name='Stage3_Phone', save_dir='checkpoints/stage3', patience=10
)

## 8. Results Summary & Export

In [ ]:
print('\n' + '='*60)
print('  TRAINING RESULTS SUMMARY')
print('='*60)
if stage1_error is not None:
    print(f'  Stage 1 (Broadcast):  {stage1_error:.2f}px')
print(f'  Stage 2 (Mixed):      {stage2_error:.2f}px')
print(f'  Stage 3 (Phone):      {stage3_error:.2f}px')
print(f'\n  Previous best: 2.41px')
print(f'  Phone data: 339 -> 946 images (+179%)')
print('='*60)

# Determine best stage
best_error = min(stage2_error, stage3_error)
best_stage = 'Stage2_Mixed' if stage2_error <= stage3_error else 'Stage3_Phone'
print(f'\n  ✅ Best model: {best_stage} ({best_error:.2f}px)')

In [ ]:
# Export to TFLite
print('Exporting best model to TFLite...')

# Load best model
best_dir = 'checkpoints/stage2' if best_stage == 'Stage2_Mixed' else 'checkpoints/stage3'
model.load_state_dict(torch.load(f'{best_dir}/best.pth'))
model.eval()

# Export to ONNX
dummy_input = torch.randn(1, 3, 256, 256).to(device)
model_cpu = model.cpu()
dummy_cpu = dummy_input.cpu()

# First get heatmap output, then add coordinate extraction
class ExportModel(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model
    
    def forward(self, x):
        heatmaps = self.base(x)  # [1, 8, 64, 64]
        B, C, H, W = heatmaps.shape
        
        # Decode: argmax per channel
        flat = heatmaps.view(B, C, -1)
        max_vals, max_idx = flat.max(dim=2)
        
        cx = (max_idx % W).float() / W  # normalize to 0-1
        cy = (max_idx // W).float() / H
        conf = torch.sigmoid(max_vals * 5)  # scale confidence
        
        # Output: [1, 24] = 8 keypoints x [x, y, confidence]
        result = torch.stack([cx, cy, conf], dim=2).view(B, -1)
        return result

export_model = ExportModel(model_cpu)
export_model.eval()

torch.onnx.export(
    export_model, dummy_cpu,
    'best_model.onnx',
    input_names=['input'],
    output_names=['output'],
    opset_version=18,
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}}
)
print('ONNX exported')

# Convert to TFLite
!pip install -q onnx2tf onnx tf2onnx sng4onnx
!onnx2tf -i best_model.onnx -o tflite_output \
    --keep_ncw_or_nchw_or_ncdhw_input_names input \
    --output_signaturedefs \
    -oiqt 2>/dev/null

# Find the float32 tflite
import glob
tflite_files = glob.glob('tflite_output/**/*.tflite', recursive=True)
print(f'TFLite files: {tflite_files}')

if tflite_files:
    # Copy best to Drive
    best_tflite = [f for f in tflite_files if 'float32' in f]
    if not best_tflite:
        best_tflite = tflite_files
    
    src = best_tflite[0]
    dst = '/content/drive/MyDrive/SHOT-AI/models/court_keypoint_v2.tflite'
    shutil.copy(src, dst)
    size_mb = os.path.getsize(src) / (1024*1024)
    print(f'\n✅ TFLite saved to Drive: {dst} ({size_mb:.1f}MB)')

In [ ]:
# Per-keypoint error analysis
model.to(device)
model.eval()

all_errors = {kp_id: [] for kp_id in KEYPOINT_IDS}

for images, heatmaps in phone_val_loader2:
    images = images.to(device)
    heatmaps = heatmaps.to(device)
    
    with torch.no_grad():
        pred = model(images)
    
    pred_coords = decode_heatmap(pred)
    gt_coords = decode_heatmap(heatmaps)
    gt_max = heatmaps.view(heatmaps.shape[0], heatmaps.shape[1], -1).max(dim=2)[0]
    
    for i, kp_id in enumerate(KEYPOINT_IDS):
        for b in range(images.shape[0]):
            if gt_max[b, i] > 0.1:
                err = torch.sqrt(((pred_coords[b, i] - gt_coords[b, i]) ** 2).sum()).item()
                all_errors[kp_id].append(err)

print('\nPer-keypoint error (pixels in 64x64 heatmap):')
print(f'{"KP":>4} | {"Mean":>6} | {"Median":>6} | {"Count":>5}')
print('-' * 35)
for kp_id in KEYPOINT_IDS:
    errs = all_errors[kp_id]
    if errs:
        mean_e = np.mean(errs)
        med_e = np.median(errs)
        print(f'{kp_id:>4} | {mean_e:>6.2f} | {med_e:>6.2f} | {len(errs):>5}')